# MCTNet data pipeline from Google Drive

Ce notebook reunit le post-traitement et la validation des tenseurs dans un seul fichier.
Il charge les CSV exportes depuis Google Earth Engine, construits les datasets finaux pour MCTNet,
puis verifie que les dimensions PyTorch sont coherentes.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    print('Google Drive monte.')
else:
    print('Google Colab non detecte. Verifie les chemins locaux si besoin.')


In [ ]:
# --- Configuration a adapter a ton Drive ---
DRIVE_BASE_DIR = Path('/content/drive/MyDrive/mctnet_crop_mapping_2021')

# Exemple attendu dans le Drive:
# /content/drive/MyDrive/mctnet_crop_mapping_2021/mctnet_samples_AR_2021.csv
# /content/drive/MyDrive/mctnet_crop_mapping_2021/mctnet_samples_CA_2021.csv
CSV_FILES = [
    DRIVE_BASE_DIR / 'mctnet_samples_AR_2021.csv',
    DRIVE_BASE_DIR / 'mctnet_samples_CA_2021.csv',
]

OUTPUT_DIR = DRIVE_BASE_DIR / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
N_TIME_STEPS = 36
TRAIN_VAL_PER_CLASS = 300
TRAIN_FRACTION_WITHIN_TRAIN_VAL = 0.8
NORMALIZE_REFLECTANCE = True
REFLECTANCE_SCALE = 10000.0
SPLIT_SEED = 2021

for csv_path in CSV_FILES:
    print(csv_path, 'exists =', csv_path.exists())

print('Output dir:', OUTPUT_DIR)


In [ ]:
def ordered_feature_columns():
    columns = []
    for time_idx in range(1, N_TIME_STEPS + 1):
        suffix = f't{time_idx:02d}'
        for band in S2_BANDS:
            columns.append(f'{band}_{suffix}')
    return columns


def ordered_valid_columns():
    return [f'valid_t{time_idx:02d}' for time_idx in range(1, N_TIME_STEPS + 1)]


def make_class_order(dataframe):
    counts = dataframe['label_final_name'].value_counts()
    ordered = [name for name in counts.index.tolist() if name != 'others']
    if 'others' in counts.index:
        ordered.append('others')
    return ordered


def make_label_mapping(dataframe):
    class_order = make_class_order(dataframe)
    name_to_index = {class_name: idx for idx, class_name in enumerate(class_order)}

    code_lookup = (
        dataframe[['label_final_name', 'label_final_code']]
        .drop_duplicates()
        .sort_values(['label_final_name', 'label_final_code'])
    )
    name_to_original_code = {
        str(row.label_final_name): int(row.label_final_code)
        for row in code_lookup.itertuples(index=False)
    }

    return name_to_index, name_to_original_code


def reshape_features(dataframe, normalize_reflectance=True, reflectance_scale=10000.0):
    feature_columns = ordered_feature_columns()
    valid_columns = ordered_valid_columns()

    x = dataframe[feature_columns].to_numpy(dtype=np.float32)
    x = x.reshape(len(dataframe), N_TIME_STEPS, len(S2_BANDS))

    if normalize_reflectance:
        x /= reflectance_scale

    valid_mask = dataframe[valid_columns].to_numpy(dtype=np.uint8)
    missing_mask = (1 - valid_mask).astype(np.uint8)
    missing_mask = np.repeat(missing_mask[:, :, None], len(S2_BANDS), axis=2)

    return x, valid_mask, missing_mask


def split_like_paper(dataframe, seed=2021):
    rng = np.random.default_rng(seed)
    train_frames = []
    val_frames = []
    test_frames = []

    train_count = int(TRAIN_VAL_PER_CLASS * TRAIN_FRACTION_WITHIN_TRAIN_VAL)
    val_count = TRAIN_VAL_PER_CLASS - train_count

    for class_name, class_frame in dataframe.groupby('label_final_name', sort=False):
        class_frame = class_frame.sample(
            frac=1.0,
            random_state=int(rng.integers(0, 2**31 - 1))
        ).reset_index(drop=True)

        if len(class_frame) < TRAIN_VAL_PER_CLASS:
            raise ValueError(
                f'Classe {class_name} avec {len(class_frame)} echantillons, '
                f'impossible d appliquer la separation du papier ({TRAIN_VAL_PER_CLASS}).'
            )

        train_val_frame = class_frame.iloc[:TRAIN_VAL_PER_CLASS].copy()
        test_frame = class_frame.iloc[TRAIN_VAL_PER_CLASS:].copy()
        train_frame = train_val_frame.iloc[:train_count].copy()
        val_frame = train_val_frame.iloc[train_count:train_count + val_count].copy()

        train_frames.append(train_frame)
        val_frames.append(val_frame)
        test_frames.append(test_frame)

    split_frames = {
        'train': pd.concat(train_frames, axis=0, ignore_index=True),
        'val': pd.concat(val_frames, axis=0, ignore_index=True),
        'test': pd.concat(test_frames, axis=0, ignore_index=True),
    }

    for split_name, split_frame in split_frames.items():
        split_frames[split_name] = split_frame.sort_values('sample_id').reset_index(drop=True)

    return split_frames


def encode_targets(dataframe, name_to_index):
    return dataframe['label_final_name'].map(name_to_index).to_numpy(dtype=np.int64)


def pack_split(dataframe, name_to_index, normalize_reflectance=True, reflectance_scale=10000.0):
    x, valid_mask, missing_mask = reshape_features(
        dataframe=dataframe,
        normalize_reflectance=normalize_reflectance,
        reflectance_scale=reflectance_scale,
    )

    return {
        'x': x.astype(np.float32),
        'valid_mask': valid_mask.astype(np.uint8),
        'missing_mask': missing_mask.astype(np.uint8),
        'y': encode_targets(dataframe, name_to_index).astype(np.int64),
        'label_final_code': dataframe['label_final_code'].to_numpy(dtype=np.int64),
        'longitude': dataframe['longitude'].to_numpy(dtype=np.float32),
        'latitude': dataframe['latitude'].to_numpy(dtype=np.float32),
        'sample_id': dataframe['sample_id'].astype(str).to_numpy(),
    }


def build_dataset_bundle(csv_path, normalize_reflectance=True, reflectance_scale=10000.0, split_seed=2021):
    dataframe = pd.read_csv(csv_path)
    dataframe = dataframe.sort_values('sample_id').reset_index(drop=True)

    required_columns = set(
        ['sample_id', 'state_name', 'label_final_name', 'label_final_code']
        + ordered_feature_columns()
        + ordered_valid_columns()
    )
    missing_columns = required_columns.difference(dataframe.columns)
    if missing_columns:
        raise ValueError(f'Colonnes manquantes dans {Path(csv_path).name}: {sorted(missing_columns)}')

    name_to_index, name_to_original_code = make_label_mapping(dataframe)
    split_frames = split_like_paper(dataframe, seed=split_seed)

    bundle = {}
    split_counts = {}

    for split_name, split_frame in split_frames.items():
        packed = pack_split(
            dataframe=split_frame,
            name_to_index=name_to_index,
            normalize_reflectance=normalize_reflectance,
            reflectance_scale=reflectance_scale,
        )
        for key, value in packed.items():
            bundle[f'{key}_{split_name}'] = value
        split_counts[split_name] = split_frame['label_final_name'].value_counts().sort_index().to_dict()

    state_name = str(dataframe['state_name'].iloc[0])
    metadata = {
        'state_name': state_name,
        'source_csv': str(csv_path),
        'n_samples_total': int(len(dataframe)),
        's2_bands': S2_BANDS,
        'n_time_steps': N_TIME_STEPS,
        'feature_shape_per_sample': [N_TIME_STEPS, len(S2_BANDS)],
        'class_name_to_index': name_to_index,
        'class_name_to_original_code': name_to_original_code,
        'split_counts': split_counts,
        'paper_settings': {
            'train_val_per_class': TRAIN_VAL_PER_CLASS,
            'train_fraction_within_train_val': TRAIN_FRACTION_WITHIN_TRAIN_VAL,
            'missing_values_marked_with_zero': True,
            'input_1_shape': [N_TIME_STEPS, len(S2_BANDS)],
            'input_2_shape': [N_TIME_STEPS],
        },
        'implementation_choices': {
            'normalize_reflectance': normalize_reflectance,
            'reflectance_scale': reflectance_scale if normalize_reflectance else None,
            'split_seed': split_seed,
        },
    }

    return bundle, metadata


def save_bundle(bundle, metadata, output_npz, output_json):
    output_npz = Path(output_npz)
    output_json = Path(output_json)
    output_npz.parent.mkdir(parents=True, exist_ok=True)
    output_json.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output_npz, **bundle)
    output_json.write_text(json.dumps(metadata, indent=2), encoding='utf-8')


built_datasets = {}
built_metadata = {}


In [ ]:
for csv_path in CSV_FILES:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f'CSV introuvable: {csv_path}')

    bundle, metadata = build_dataset_bundle(
        csv_path=csv_path,
        normalize_reflectance=NORMALIZE_REFLECTANCE,
        reflectance_scale=REFLECTANCE_SCALE,
        split_seed=SPLIT_SEED,
    )

    state_slug = metadata['state_name'].lower().replace(' ', '_')
    output_npz = OUTPUT_DIR / f'{state_slug}_mctnet_dataset.npz'
    output_json = OUTPUT_DIR / f'{state_slug}_mctnet_dataset.json'
    save_bundle(bundle, metadata, output_npz, output_json)

    built_datasets[state_slug] = bundle
    built_metadata[state_slug] = metadata

    print(f'[{metadata["state_name"]}] dataset ecrit: {output_npz}')
    print(f'[{metadata["state_name"]}] metadonnees: {output_json}')
    print(json.dumps(metadata['split_counts'], indent=2))
    print('-' * 80)


In [ ]:
class CropTimeSeriesDataset(Dataset):
    def __init__(self, bundle, split):
        self.x = torch.from_numpy(bundle[f'x_{split}']).float()
        self.valid_mask = torch.from_numpy(bundle[f'valid_mask_{split}']).float()
        self.missing_mask = torch.from_numpy(bundle[f'missing_mask_{split}']).float()
        self.y = torch.from_numpy(bundle[f'y_{split}']).long()

    def __len__(self):
        return int(self.x.shape[0])

    def __getitem__(self, index):
        return {
            'x': self.x[index],
            'valid_mask': self.valid_mask[index],
            'missing_mask': self.missing_mask[index],
            'y': self.y[index],
        }


def load_bundle(npz_path):
    with np.load(npz_path, allow_pickle=True) as data:
        return {key: data[key] for key in data.files}


def validate_split_shapes(bundle, split):
    x = bundle[f'x_{split}']
    valid_mask = bundle[f'valid_mask_{split}']
    missing_mask = bundle[f'missing_mask_{split}']
    y = bundle[f'y_{split}']

    assert x.ndim == 3, f'{split}: x doit etre 3D, recu {x.shape}'
    assert x.shape[1:] == (36, 10), f'{split}: x attendu [N, 36, 10], recu {x.shape}'
    assert valid_mask.shape == (x.shape[0], 36), f'{split}: valid_mask attendu [N, 36], recu {valid_mask.shape}'
    assert missing_mask.shape == (x.shape[0], 36, 10), f'{split}: missing_mask attendu [N, 36, 10], recu {missing_mask.shape}'
    assert y.shape == (x.shape[0],), f'{split}: y attendu [N], recu {y.shape}'

    recovered_missing_mask = np.repeat((1 - valid_mask)[:, :, None], 10, axis=2)
    assert np.array_equal(missing_mask, recovered_missing_mask), f'{split}: missing_mask ne correspond pas a 1 - valid_mask'


def validate_dataloader(bundle, split, batch_size=32):
    dataset = CropTimeSeriesDataset(bundle, split)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    batch = next(iter(loader))

    assert batch['x'].shape[1:] == (36, 10)
    assert batch['valid_mask'].shape[1:] == (36,)
    assert batch['missing_mask'].shape[1:] == (36, 10)
    assert batch['y'].ndim == 1

    print(
        f'[{split}] batch OK | '
        f'x={tuple(batch["x"].shape)} | '
        f'valid_mask={tuple(batch["valid_mask"].shape)} | '
        f'missing_mask={tuple(batch["missing_mask"].shape)} | '
        f'y={tuple(batch["y"].shape)}'
    )


In [ ]:
dataset_files = sorted(OUTPUT_DIR.glob('*_mctnet_dataset.npz'))
metadata_files = sorted(OUTPUT_DIR.glob('*_mctnet_dataset.json'))

print('Datasets trouves:')
for file_path in dataset_files:
    print(' -', file_path)

print('Metadata trouves:')
for file_path in metadata_files:
    print(' -', file_path)

for npz_path in dataset_files:
    state_slug = npz_path.stem.replace('_mctnet_dataset', '')
    json_path = OUTPUT_DIR / f'{state_slug}_mctnet_dataset.json'
    bundle = load_bundle(npz_path)
    metadata = json.loads(json_path.read_text(encoding='utf-8'))

    print('=' * 80)
    print('State:', metadata['state_name'])
    print('Classes:', metadata['class_name_to_index'])

    for split in ['train', 'val', 'test']:
        validate_split_shapes(bundle, split)
        validate_dataloader(bundle, split, batch_size=32)

print('=' * 80)
print('Toutes les dimensions attendues par le pipeline MCTNet sont correctes.')
